In [1]:
# Cell 1 -- paths + settings. Exports run on the CPU (no GPU memory needed) and do not touch any training files.
import sys, json, shutil, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn

PROJ = Path(r"C:\Users\LENOVO\Documents\Thermal_Fusion_Paper")
NANODET_REPO = Path(r"C:\Users\LENOVO\Documents\NanoDet")
sys.path.insert(0, str(PROJ))
import lb_fusion as L

BUNDLE = L.JOURNAL / "jetson_bundle"
ONNX_DIR = BUNDLE / "onnx"
ONNX_DIR.mkdir(parents=True, exist_ok=True)
OPSET = 17
CPU = torch.device("cpu")
print(f"bundle folder: {BUNDLE}")

bundle folder: C:\Users\LENOVO\Documents\Thermal_Fusion_Paper\journal\jetson_bundle


In [2]:
# Cell 2 -- YOLOv8n models (FLIR, 3 classes, letterboxed 640x640). Output: (1, 7, 8400) raw head output; NMS runs on the Jetson.
class YoloExport(nn.Module):
    def __init__(self, model, kind):
        super().__init__()
        self.m, self.kind = model, kind

    def forward(self, *x):
        if self.kind == "A":                       # RGB only
            return self.m.base(x[0])[0]
        if self.kind == "T":                       # thermal only, 1 channel repeated to 3
            return self.m.base(x[0].repeat(1, 3, 1, 1))[0]
        return self.m(x[0], x[1])[0]               # frozen RGB + thermal adapter


ck = PROJ / "checkpoints_lb"
YOLO_MODELS = {
    "yolo_A":           ("A",  ("A", L.DATASETS["flir"]["config_a"])),
    "yolo_B_s1":        ("RT", ("fusion", "B", ck / "lb_yolo_flir_B_s1_best.pt")),
    "yolo_C_s1":        ("RT", ("fusion", "C", ck / "lb_yolo_flir_C_s1_best.pt")),
    "yolo_B_robust_s1": ("RT", ("fusion", "B_robust", ck / "lb_yolo_flir_B_robust_s1_best.pt")),
    "yolo_T":           ("T",  ("thermal_only", PROJ / r"results\lb_yolo_flir_T_thermalonly_s1\weights\best.pt")),
}

manifest = {"models": {}}
REFS = {}
rgb640 = torch.rand(1, 3, 640, 640)
th640 = torch.rand(1, 1, 640, 640)
for name, (kind, spec) in YOLO_MODELS.items():
    model = L._load_eval_model(spec, "flir", CPU).eval()
    wrap = YoloExport(model, kind).eval()
    if kind == "A":
        args, names = (rgb640,), ["rgb"]
    elif kind == "T":
        args, names = (th640,), ["thermal"]
    else:
        args, names = (rgb640, th640), ["rgb", "thermal"]
    out_path = ONNX_DIR / f"{name}.onnx"
    with torch.no_grad():
        ref = wrap(*args).numpy()
        torch.onnx.export(wrap, args, str(out_path), input_names=names, output_names=["output"],
                          opset_version=OPSET, do_constant_folding=True, dynamo=False)
    manifest["models"][name] = {"arch": "yolov8n", "inputs": names, "input_size": 640,
                                "output_shape": list(ref.shape), "kind": kind}
    REFS[name] = (args, ref)
    print(f"exported {out_path.name}: inputs {names}, output {ref.shape}, {out_path.stat().st_size / 1e6:.1f} MB")

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_35172\1934747238.py:40: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(wrap, args, str(out_path), input_names=names, output_names=["output"],


exported yolo_A.onnx: inputs ['rgb'], output (1, 7, 8400), 12.3 MB
exported yolo_B_s1.onnx: inputs ['rgb', 'thermal'], output (1, 7, 8400), 12.4 MB
exported yolo_C_s1.onnx: inputs ['rgb', 'thermal'], output (1, 7, 8400), 12.6 MB
exported yolo_B_robust_s1.onnx: inputs ['rgb', 'thermal'], output (1, 7, 8400), 12.4 MB
exported yolo_T.onnx: inputs ['thermal'], output (1, 7, 8400), 12.3 MB


In [3]:
# Cell 3 -- NanoDet-Plus models (FLIR, 416x416 stretched, BGR ImageNet-normalised RGB, thermal /255). Latency only on the Jetson.
sys.path.insert(0, str(NANODET_REPO))
from nanodet.util import cfg, load_config
from nanodet.model.arch import build_model as nd_build_model


def nd_branch(out_channels, n_extra=0, mid=16):
    chans = [mid, mid * 2] + [mid * 2] * n_extra + [out_channels]
    layers, c_in = [], 1
    for c_out in chans:
        layers += [nn.Conv2d(c_in, c_out, 3, 2, 1), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True)]
        c_in = c_out
    m = nn.Module()
    m.net = nn.Sequential(*layers)
    return m


class NanoDetExport(nn.Module):
    def __init__(self, kind, fusion_ckpt=None):
        super().__init__()
        load_config(cfg, str(PROJ / r"configs\model_only_config.yml"))
        base = nd_build_model(cfg.model)
        state = torch.load(PROJ / r"results\nanodet_config_a_flir\model_best\model_best.ckpt", map_location="cpu")
        base.load_state_dict({k.replace("avg_model.", "", 1): v for k, v in state["state_dict"].items()
                              if k.startswith("avg_model.")}, strict=True)
        self.base, self.kind, self._x = base.eval(), kind, None
        if kind == "A":
            return
        ckf = torch.load(fusion_ckpt, map_location="cpu")
        self.tb0, self.fm0 = nd_branch(116), L.ChannelAttentionFusion(116)
        if kind == "B":
            self.tb0.load_state_dict(ckf["thermal_branch"]); self.fm0.load_state_dict(ckf["fusion_module"])
        else:
            self.tb1, self.fm1 = nd_branch(232, n_extra=1), L.ChannelAttentionFusion(232)
            self.tb0.load_state_dict(ckf["thermal_branch_0"]); self.fm0.load_state_dict(ckf["fusion_module_0"])
            self.tb1.load_state_dict(ckf["thermal_branch_1"]); self.fm1.load_state_dict(ckf["fusion_module_1"])
        self.base.backbone.register_forward_hook(self._hook)

    def _hook(self, module, inputs, output):
        f0, f1, f2 = output
        f0 = self.fm0(f0, self.tb0.net(self._x))
        if self.kind == "C":
            f1 = self.fm1(f1, self.tb1.net(self._x))
        return (f0, f1, f2)

    def forward(self, *x):
        if self.kind != "A":
            self._x = x[1]
        return self.base(x[0])


NANO_MODELS = {
    "nanodet_A":    ("A", None),
    "nanodet_B_s1": ("B", PROJ / "checkpoints" / "config_b_nanodet_single_fusion_best.pt"),
    "nanodet_C_s1": ("C", PROJ / "checkpoints" / "config_c_nanodet_multi_fusion_best.pt"),
}
rgb416 = torch.randn(1, 3, 416, 416)
th416 = torch.rand(1, 1, 416, 416)
for name, (kind, fck) in NANO_MODELS.items():
    wrap = NanoDetExport(kind, fck).eval()
    args, names = ((rgb416,), ["rgb"]) if kind == "A" else ((rgb416, th416), ["rgb", "thermal"])
    out_path = ONNX_DIR / f"{name}.onnx"
    with torch.no_grad():
        ref = wrap(*args).numpy()
        torch.onnx.export(wrap, args, str(out_path), input_names=names, output_names=["output"],
                          opset_version=OPSET, do_constant_folding=True, dynamo=False)
    manifest["models"][name] = {"arch": "nanodet-plus-m", "inputs": names, "input_size": 416,
                                "output_shape": list(ref.shape), "kind": kind}
    REFS[name] = (args, ref)
    print(f"exported {out_path.name}: inputs {names}, output {ref.shape}, {out_path.stat().st_size / 1e6:.1f} MB")

C:\Users\LENOVO\miniconda3\envs\cv_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


model size is  1.0x
init weights...
Finish initialize NanoDet-Plus Head.


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_35172\2243155529.py:65: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(wrap, args, str(out_path), input_names=names, output_names=["output"],


exported nanodet_A.onnx: inputs ['rgb'], output (1, 3598, 35), 4.7 MB
model size is  1.0x
init weights...
Finish initialize NanoDet-Plus Head.
exported nanodet_B_s1.onnx: inputs ['rgb', 'thermal'], output (1, 3598, 35), 4.9 MB
model size is  1.0x
init weights...
Finish initialize NanoDet-Plus Head.
exported nanodet_C_s1.onnx: inputs ['rgb', 'thermal'], output (1, 3598, 35), 5.4 MB


In [4]:
# Cell 4 -- check every ONNX file against PyTorch on the same random input (onnxruntime, CPU)
import onnxruntime as ort
for name, (args, ref) in REFS.items():
    sess = ort.InferenceSession(str(ONNX_DIR / f"{name}.onnx"), providers=["CPUExecutionProvider"])
    feeds = {i.name: a.numpy() for i, a in zip(sess.get_inputs(), args)}
    out = sess.run(None, feeds)[0]
    if name.startswith("nanodet"):
        # NanoDet's own ONNX path applies the sigmoid to the class scores inside the graph
        nc = len(L.DATASETS["flir"]["names"])
        ref = ref.copy()
        ref[..., :nc] = 1 / (1 + np.exp(-ref[..., :nc]))
    err = float(np.abs(out - ref).max())
    print(f"{name:<18} max |onnx - torch| = {err:.2e}  {'OK' if err < 1e-2 else 'CHECK THIS'}")

yolo_A             max |onnx - torch| = 5.04e-04  OK
yolo_B_s1          max |onnx - torch| = 8.24e-04  OK
yolo_C_s1          max |onnx - torch| = 5.49e-04  OK
yolo_B_robust_s1   max |onnx - torch| = 6.10e-04  OK
yolo_T             max |onnx - torch| = 3.72e-03  OK
nanodet_A          max |onnx - torch| = 7.87e-06  OK
nanodet_B_s1       max |onnx - torch| = 9.54e-06  OK
nanodet_C_s1       max |onnx - torch| = 1.00e-05  OK


In [5]:
# Cell 5 -- images for the Jetson: all 1013 FLIR val pairs (accuracy) + 300 random FLIR train pairs (INT8 calibration only)
ds = L.DATASETS["flir"]
fit_ids, _, eval_sets = L.get_splits("flir", smoke=False)
val_ids, night_ids = eval_sets["full"], eval_sets["night"]
calib_ids = sorted(random.Random(0).sample(fit_ids, 300))

def copy_pairs(ids, split, dst):
    (dst / "rgb").mkdir(parents=True, exist_ok=True)
    (dst / "thermal").mkdir(parents=True, exist_ok=True)
    for i in ids:
        name = f"{i}{ds['ext']}"
        if not (dst / "rgb" / name).exists():
            shutil.copy2(ds["root"] / "images" / split / name, dst / "rgb" / name)
        if not (dst / "thermal" / name).exists():
            shutil.copy2(ds["thermal"][split] / name, dst / "thermal" / name)

copy_pairs(val_ids, "val", BUNDLE / "data" / "val")
copy_pairs(calib_ids, "train", BUNDLE / "data" / "calib")
(BUNDLE / "data" / "val_ids.json").write_text(json.dumps(val_ids))
(BUNDLE / "data" / "night_ids.json").write_text(json.dumps(night_ids))
(BUNDLE / "data" / "calib_ids.json").write_text(json.dumps(calib_ids))

manifest.update({
    "dataset": "flir", "class_names": ds["names"], "image_ext": ds["ext"],
    "yolo_preprocess": "letterbox to 640 (pad 114), RGB, /255; thermal letterboxed identically (pad 114), /255",
    "nanodet_preprocess": "resize (stretch) to 416, BGR, (x - [103.53,116.28,123.675]) / [57.375,57.12,58.395]; thermal resize, /255",
    "nms": {"conf": 0.001, "iou": 0.6, "max_det": 300},
})
(BUNDLE / "manifest.json").write_text(json.dumps(manifest, indent=2))
size = sum(f.stat().st_size for f in BUNDLE.rglob("*") if f.is_file()) / 1e6
print(f"val pairs: {len(val_ids)}  calib pairs: {len(calib_ids)}  bundle size: {size:.0f} MB")
print(f"copy this whole folder to the Jetson: {BUNDLE}")

loaded holdout split: C:\Users\LENOVO\Documents\Thermal_Fusion_Paper\journal\splits\flir_train_holdout.json
val pairs: 1013  calib pairs: 300  bundle size: 326 MB
copy this whole folder to the Jetson: C:\Users\LENOVO\Documents\Thermal_Fusion_Paper\journal\jetson_bundle
